In [1]:
# =============================================================================
# NOTEBOOK 6: FULL FRAMEWORK EVALUATION & PAPER RESULTS
# =============================================================================
#
# PURPOSE:
#   1. Run the COMPLETE framework: DP-SGD + Byzantine-resilient aggregation
#      (Krum and Trimmed Mean) simultaneously
#   2. Final evaluation on 500 test samples for key configurations
#   3. Consolidate ALL results from Notebooks 1-5 into the paper's tables
#
# NOTE: Secure aggregation (Notebook 4) does not change model weights —
# it is a communication protocol that produces mathematically identical
# aggregates. So "FedAvg+DP+SecAgg" yields the same model as "FedAvg+DP".
# SecAgg overhead is reported from Notebook 4 benchmarks.
#
# EXPERIMENTS:
#   - DP + Krum (eps=8, 2 Byzantine sign-flip, alpha=0.5)
#   - DP + Trimmed Mean (eps=8, 2 Byzantine sign-flip, alpha=0.5)
#   - DP + Krum (eps=8, 1 Byzantine sign-flip, alpha=0.5)
#   - DP + Trimmed Mean (eps=8, 1 Byzantine sign-flip, alpha=0.5)
#   - Final 500-sample evaluation on best configs from each notebook
#
# HARDWARE: Colab Pro H100, native bf16 + SDPA
# ESTIMATED RUNTIME: 3-5 hours
# =============================================================================


# =============================================================================
# PHASE 1: SETUP
# =============================================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

sys.setrecursionlimit(10000)

OUTPUT_DIR = "/content/drive/MyDrive/uav_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Installing packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy==2.0.2", "h5py"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "peft", "accelerate",
    "datasets", "trl", "nltk", "rouge-score",
    "sentencepiece", "protobuf", "triton"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "bitsandbytes>=0.43.0", "--force-reinstall", "--no-deps"])
print("All packages installed.")

# --- Dataset ---
DRIVE_ZIP = "/content/drive/MyDrive/uav_dataset.zip"
DATASET_DIR = "/content/dataset"

if os.path.exists(DRIVE_ZIP):
    print("Extracting dataset...")
    os.makedirs(DATASET_DIR, exist_ok=True)
    subprocess.run(["unzip", "-qo", DRIVE_ZIP, "-d", DATASET_DIR])

DATASET_PATH = None
for root, dirs, files in os.walk(DATASET_DIR):
    if "train.json" in files:
        DATASET_PATH = root
        break
print(f"Dataset: {DATASET_PATH}")


# =============================================================================
# PHASE 2: IMPORTS AND CONFIGURATION
# =============================================================================

import json, time, copy, random, gc, math
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter, OrderedDict

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments, TrainerCallback,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("All imports successful.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- Model Config ----
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LENGTH = 512
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ---- Federated Config (H100 native bf16) ----
NUM_CLIENTS = 5
NUM_ROUNDS = 20
LOCAL_EPOCHS = 1
LOCAL_BATCH_SIZE = 64
LOCAL_GRADIENT_ACCUMULATION = 1
LOCAL_LEARNING_RATE = 5e-4
DIRICHLET_ALPHA = 0.5

# ---- DP Config ----
DP_GRAD_CLIP_NORM = 1.0
TARGET_EPSILON = 8
DP_DELTA = 1e-5

# ---- Byzantine Config ----
SIGN_FLIP_SCALE = 2.0

# ---- Evaluation Config ----
NUM_EVAL_SAMPLES_INTERMEDIATE = 100
NUM_EVAL_SAMPLES_FINAL = 500
MAX_NEW_TOKENS = 512
EVAL_EVERY_N_ROUNDS = 5

print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# =============================================================================
# PHASE 3: LOAD DATASET
# =============================================================================

def load_json(filepath):
    with open(filepath, 'r') as f:
        return json.load(f)

def format_sample(sample):
    input_text = sample["input"]
    output_json = json.dumps(sample["output"], indent=None, ensure_ascii=False)
    return (
        f"<|system|>\n"
        f"You are a UAV command interpreter. Convert the natural language "
        f"command into a structured JSON command for drone execution.</s>\n"
        f"<|user|>\n"
        f"{input_text}</s>\n"
        f"<|assistant|>\n"
        f"{output_json}</s>"
    )

train_raw = load_json(os.path.join(DATASET_PATH, "train.json"))
val_raw = load_json(os.path.join(DATASET_PATH, "val.json"))
test_raw = load_json(os.path.join(DATASET_PATH, "test.json"))
print(f"Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")


# =============================================================================
# PHASE 4: ALL HELPER FUNCTIONS
# =============================================================================

def dirichlet_split(data, n_clients, alpha, seed=42):
    np.random.seed(seed)
    type_indices = {}
    for i, sample in enumerate(data):
        ct = sample["output"]["command_type"]
        if ct not in type_indices:
            type_indices[ct] = []
        type_indices[ct].append(i)
    client_indices = [[] for _ in range(n_clients)]
    for ct, indices in type_indices.items():
        np.random.shuffle(indices)
        proportions = np.random.dirichlet([alpha] * n_clients)
        splits = (proportions * len(indices)).astype(int)
        splits[-1] = len(indices) - splits[:-1].sum()
        start = 0
        for c, count in enumerate(splits):
            client_indices[c].extend(indices[start:start + count])
            start += count
    client_datasets = []
    for indices in client_indices:
        np.random.shuffle(indices)
        client_datasets.append([data[i] for i in indices])
    return client_datasets


def get_lora_state_dict(model):
    lora_state = OrderedDict()
    for name, param in model.named_parameters():
        if "lora_" in name:
            lora_state[name] = param.data.clone().cpu()
    return lora_state


def set_lora_state_dict(model, state_dict):
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.data.copy_(state_dict[name].to(param.device))


def generate_prediction(input_text, model, tokenizer):
    prompt = (
        f"<|system|>\n"
        f"You are a UAV command interpreter. Convert the natural language "
        f"command into a structured JSON command for drone execution.</s>\n"
        f"<|user|>\n"
        f"{input_text}</s>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    if "</s>" in text:
        text = text.split("</s>")[0].strip()
    return text


def evaluate_model(model, tokenizer, test_data, num_samples=100):
    model.eval()
    smoother = SmoothingFunction().method1
    eval_samples = test_data[:num_samples]
    bleu_scores, valid_json, cmd_type_match, action_match, valid_count = [], 0, 0, 0, 0

    for idx, sample in enumerate(eval_samples):
        pred_text = generate_prediction(sample["input"], model, tokenizer)
        ref_text = json.dumps(sample["output"], indent=None, ensure_ascii=False)
        try:
            score = sentence_bleu([list(ref_text)], list(pred_text),
                                  smoothing_function=smoother)
            bleu_scores.append(score)
        except:
            bleu_scores.append(0.0)
        try:
            text = pred_text.strip()
            s, e = text.find("{"), text.rfind("}")
            if s != -1 and e != -1:
                parsed = json.loads(text[s:e+1])
                valid_json += 1
                valid_count += 1
                if parsed.get("command_type") == sample["output"].get("command_type"):
                    cmd_type_match += 1
                if parsed.get("action") == sample["output"].get("action"):
                    action_match += 1
        except:
            pass
        if (idx + 1) % 50 == 0:
            print(f"    Eval: {idx+1}/{num_samples}", flush=True)

    n = len(eval_samples)
    vc = max(valid_count, 1)
    return {
        "bleu": round(np.mean(bleu_scores), 4),
        "json_validity": round(valid_json / n * 100, 1),
        "cmd_type_acc": round(cmd_type_match / vc * 100, 1),
        "action_acc": round(action_match / vc * 100, 1),
    }


# =============================================================================
# PHASE 5: DP NOISE CALLBACK (same as Notebook 3 v3)
# =============================================================================

class DPNoiseCallback(TrainerCallback):
    """Injects calibrated Gaussian noise into LoRA params after each step."""
    def __init__(self, noise_std, model):
        self.noise_std = noise_std
        self.model = model
        self.steps_noised = 0

    def on_step_end(self, args, state, control, **kwargs):
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if "lora_" in name and param.requires_grad:
                    noise = torch.randn_like(param) * self.noise_std
                    param.add_(noise)
        self.steps_noised += 1


def compute_noise_multiplier(target_epsilon, delta, total_steps, sampling_rate):
    sigma_low, sigma_high = 0.01, 500.0
    for _ in range(200):
        sigma_mid = (sigma_low + sigma_high) / 2.0
        eps = _compute_rdp_epsilon(sigma_mid, total_steps, sampling_rate, delta)
        if eps < target_epsilon:
            sigma_high = sigma_mid
        else:
            sigma_low = sigma_mid
        if abs(sigma_high - sigma_low) < 1e-5:
            break
    return sigma_high


def _compute_rdp_epsilon(sigma, total_steps, sampling_rate, delta):
    alphas = [1.25, 1.5, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 32, 64]
    min_epsilon = float('inf')
    for alpha in alphas:
        rdp_per_step = (sampling_rate ** 2) * alpha / (2.0 * sigma ** 2)
        rdp_total = rdp_per_step * total_steps
        epsilon = rdp_total + math.log(1.0 / delta) / (alpha - 1.0)
        min_epsilon = min(min_epsilon, epsilon)
    return min_epsilon


def compute_per_step_noise_std(sigma, grad_clip_norm, learning_rate, effective_batch_size):
    return learning_rate * sigma * grad_clip_norm / effective_batch_size


# =============================================================================
# PHASE 6: AGGREGATION RULES
# =============================================================================

def fedavg_aggregate(client_state_dicts, client_sizes):
    total_size = sum(client_sizes)
    avg_state = OrderedDict()
    for key in client_state_dicts[0]:
        avg_state[key] = torch.zeros_like(client_state_dicts[0][key], dtype=torch.float32)
    for state_dict, size in zip(client_state_dicts, client_sizes):
        weight = size / total_size
        for key in avg_state:
            avg_state[key] += state_dict[key].float() * weight
    return avg_state


def krum_aggregate(client_state_dicts, client_sizes, num_byzantine):
    n = len(client_state_dicts)
    f = num_byzantine
    flat_updates = []
    for sd in client_state_dicts:
        flat = torch.cat([v.float().flatten() for v in sd.values()])
        flat_updates.append(flat)

    distances = torch.zeros(n, n)
    for i in range(n):
        for j in range(i + 1, n):
            d = torch.norm(flat_updates[i] - flat_updates[j], p=2).item() ** 2
            distances[i][j] = d
            distances[j][i] = d

    num_neighbors = max(1, n - f - 2)
    scores = []
    for i in range(n):
        dists_i = distances[i].clone()
        dists_i[i] = float('inf')
        nearest = torch.topk(dists_i, num_neighbors, largest=False).values
        scores.append(nearest.sum().item())

    selected = int(np.argmin(scores))
    print(f"    Krum selected client {selected}")

    result = OrderedDict()
    for key in client_state_dicts[selected]:
        result[key] = client_state_dicts[selected][key].float().clone()
    return result


def trimmed_mean_aggregate(client_state_dicts, client_sizes, num_byzantine):
    n = len(client_state_dicts)
    f = num_byzantine
    keys = list(client_state_dicts[0].keys())
    result = OrderedDict()
    for key in keys:
        stacked = torch.stack([sd[key].float() for sd in client_state_dicts])
        original_shape = stacked.shape[1:]
        flat = stacked.reshape(n, -1)
        sorted_vals, _ = torch.sort(flat, dim=0)
        if f > 0 and n > 2 * f:
            trimmed = sorted_vals[f:n - f, :]
        else:
            trimmed = sorted_vals
        result[key] = trimmed.mean(dim=0).reshape(original_shape)
    return result


# =============================================================================
# PHASE 7: BYZANTINE ATTACK
# =============================================================================

def apply_sign_flip_attack(honest_state, global_state):
    poisoned = OrderedDict()
    for key in honest_state:
        delta = honest_state[key].float() - global_state[key].float()
        poisoned[key] = global_state[key].float() - SIGN_FLIP_SCALE * delta
    return poisoned


# =============================================================================
# PHASE 8: FULL FRAMEWORK TRAINING LOOP (DP + Byzantine)
# =============================================================================

def run_full_framework_experiment(model, tokenizer, client_datasets, test_raw,
                                   num_byzantine, aggregation_method,
                                   target_epsilon, num_rounds, local_epochs):
    """
    Combined DP + Byzantine resilience experiment.
    - Per-step DP noise injection via callback (privacy)
    - Byzantine clients use sign-flip attack (strongest generic)
    - Aggregation via Krum or Trimmed Mean (defense)
    """

    config_str = f"full_byz{num_byzantine}_{aggregation_method}_eps{target_epsilon}"
    print(f"\n{'='*60}")
    print(f"FULL FRAMEWORK: {config_str}")
    print(f"{'='*60}")

    n_clients = len(client_datasets)
    effective_batch = LOCAL_BATCH_SIZE * LOCAL_GRADIENT_ACCUMULATION

    # Compute DP noise parameters
    avg_client_size = int(np.mean([len(cd) for cd in client_datasets]))
    steps_per_epoch = max(1, avg_client_size // effective_batch)
    total_steps_per_client = steps_per_epoch * local_epochs * num_rounds
    sampling_rate = effective_batch / avg_client_size

    sigma = compute_noise_multiplier(target_epsilon, DP_DELTA,
                                      total_steps_per_client, sampling_rate)
    param_noise_std = compute_per_step_noise_std(
        sigma, DP_GRAD_CLIP_NORM, LOCAL_LEARNING_RATE, effective_batch
    )
    achieved_eps = _compute_rdp_epsilon(sigma, total_steps_per_client,
                                         sampling_rate, DP_DELTA)

    print(f"  Byzantine: {num_byzantine}/{n_clients} (sign-flip)")
    print(f"  Aggregation: {aggregation_method}")
    print(f"  DP: sigma={sigma:.6f} | noise_std={param_noise_std:.8f}")
    print(f"  Achieved epsilon: {achieved_eps:.2f} (target: {target_epsilon})")

    global_state = get_lora_state_dict(model)

    history = {
        "rounds": [], "train_losses": [], "eval_metrics": [],
        "dp_config": {
            "target_epsilon": target_epsilon,
            "achieved_epsilon": round(achieved_eps, 2),
            "delta": DP_DELTA,
            "sigma": round(sigma, 6),
            "param_noise_std": round(param_noise_std, 8),
        }
    }

    experiment_start = time.time()

    for round_num in range(1, num_rounds + 1):
        round_start = time.time()

        client_states = []
        client_sizes = []
        client_losses = []

        for c_idx in range(n_clients):
            c_data = client_datasets[c_idx]
            is_byzantine = c_idx < num_byzantine

            set_lora_state_dict(model, global_state)

            c_formatted = [{"text": format_sample(s)} for s in c_data]
            c_dataset = Dataset.from_list(c_formatted)

            # DP noise callback
            dp_callback = DPNoiseCallback(noise_std=param_noise_std, model=model)

            local_args = TrainingArguments(
                output_dir=os.path.join("/content", f"tmp_full_c{c_idx}"),
                num_train_epochs=local_epochs,
                per_device_train_batch_size=LOCAL_BATCH_SIZE,
                gradient_accumulation_steps=LOCAL_GRADIENT_ACCUMULATION,
                learning_rate=LOCAL_LEARNING_RATE,
                warmup_steps=10,
                bf16=True,
                max_grad_norm=DP_GRAD_CLIP_NORM,
                optim="adamw_torch",
                logging_steps=9999,
                save_strategy="no",
                report_to="none",
                seed=SEED + round_num + c_idx,
                dataloader_num_workers=0,
                remove_unused_columns=True,
            )

            local_trainer = SFTTrainer(
                model=model,
                args=local_args,
                train_dataset=c_dataset,
                processing_class=tokenizer,
                callbacks=[dp_callback],
            )

            local_result = local_trainer.train()
            client_losses.append(local_result.training_loss)

            trained_state = get_lora_state_dict(model)

            # Apply sign-flip attack if Byzantine
            if is_byzantine and num_byzantine > 0:
                poisoned_state = apply_sign_flip_attack(trained_state, global_state)
                client_states.append(poisoned_state)
            else:
                client_states.append(trained_state)

            client_sizes.append(len(c_data))

            del local_trainer, local_args, c_dataset, c_formatted, dp_callback
            gc.collect()
            torch.cuda.empty_cache()

        # Byzantine-resilient aggregation
        if aggregation_method == "fedavg":
            global_state = fedavg_aggregate(client_states, client_sizes)
        elif aggregation_method == "krum":
            global_state = krum_aggregate(client_states, client_sizes, num_byzantine)
        elif aggregation_method == "trimmed_mean":
            global_state = trimmed_mean_aggregate(client_states, client_sizes, num_byzantine)

        set_lora_state_dict(model, global_state)
        del client_states
        gc.collect()

        round_time = time.time() - round_start
        avg_loss = np.mean(client_losses)

        history["rounds"].append(round_num)
        history["train_losses"].append(round(float(avg_loss), 4))

        if round_num % EVAL_EVERY_N_ROUNDS == 0 or round_num == num_rounds:
            print(f"\n  Round {round_num}/{num_rounds} | Loss: {avg_loss:.4f} | "
                  f"Time: {round_time:.0f}s")
            print(f"  Evaluating...", flush=True)

            metrics = evaluate_model(model, tokenizer, test_raw,
                                      NUM_EVAL_SAMPLES_INTERMEDIATE)
            history["eval_metrics"].append({"round": round_num, **metrics})
            print(f"  BLEU={metrics['bleu']:.4f} | JSON={metrics['json_validity']}% | "
                  f"CmdType={metrics['cmd_type_acc']}% | Action={metrics['action_acc']}%")
        else:
            print(f"  Round {round_num}/{num_rounds} | Loss: {avg_loss:.4f} | "
                  f"Time: {round_time:.0f}s")

    # FINAL EVALUATION on 500 samples
    print(f"\n  Running FINAL evaluation on {NUM_EVAL_SAMPLES_FINAL} test samples...")
    final_metrics = evaluate_model(model, tokenizer, test_raw, NUM_EVAL_SAMPLES_FINAL)
    history["final_eval_500"] = final_metrics
    print(f"  FINAL (500): BLEU={final_metrics['bleu']:.4f} | "
          f"JSON={final_metrics['json_validity']}% | "
          f"CmdType={final_metrics['cmd_type_acc']}% | "
          f"Action={final_metrics['action_acc']}%")

    total_time = time.time() - experiment_start
    history["total_time_minutes"] = round(total_time / 60, 1)
    return history


# =============================================================================
# PHASE 9: LOAD MODEL (Native bf16 + SDPA for H100)
# =============================================================================

print("\n" + "=" * 60)
print("Loading model (bf16 + SDPA for H100)...")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable (LoRA): {trainable_params:,}")


# =============================================================================
# PHASE 10: PREPARE DATA
# =============================================================================

print("\nSplitting data...")
client_datasets = dirichlet_split(train_raw, NUM_CLIENTS, DIRICHLET_ALPHA)
for c, cd in enumerate(client_datasets):
    print(f"  Client {c}: {len(cd)} samples")


# =============================================================================
# PHASE 11: RUN FULL FRAMEWORK EXPERIMENTS
# =============================================================================

print("\n" + "=" * 60)
print("RUNNING FULL FRAMEWORK EXPERIMENTS (DP + Byzantine)")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")

full_results_path = os.path.join(OUTPUT_DIR, "full_framework_results.json")
if os.path.exists(full_results_path):
    with open(full_results_path, 'r') as f:
        all_full_results = json.load(f)
    clean = {}
    for key, val in all_full_results.items():
        if val.get("final_eval_500") and val["final_eval_500"].get("bleu", 0) > 0:
            clean[key] = val
            print(f"  Loaded: {key}")
    all_full_results = clean
else:
    all_full_results = {}

# Experiment list: DP + Byzantine defense
full_experiments = [
    # (num_byzantine, aggregation, description)
    (0, "fedavg",        "DP only (no Byzantine)"),
    (1, "krum",          "DP + Krum (1 Byz)"),
    (1, "trimmed_mean",  "DP + TrimmedMean (1 Byz)"),
    (2, "krum",          "DP + Krum (2 Byz)"),
    (2, "trimmed_mean",  "DP + TrimmedMean (2 Byz)"),
    (2, "fedavg",        "DP + FedAvg (2 Byz, no defense)"),
]

for num_byz, agg, desc in full_experiments:
    exp_key = f"full_byz{num_byz}_{agg}_eps{TARGET_EPSILON}"

    if exp_key in all_full_results:
        print(f"\n  {exp_key} already done. Skipping.")
        continue

    print(f"\n  Starting: {desc}")

    # Reset LoRA
    for name, param in model.named_parameters():
        if "lora_A" in name:
            torch.nn.init.kaiming_uniform_(param, a=np.sqrt(5))
        elif "lora_B" in name:
            torch.nn.init.zeros_(param)

    history = run_full_framework_experiment(
        model=model, tokenizer=tokenizer,
        client_datasets=client_datasets, test_raw=test_raw,
        num_byzantine=num_byz, aggregation_method=agg,
        target_epsilon=TARGET_EPSILON,
        num_rounds=NUM_ROUNDS, local_epochs=LOCAL_EPOCHS,
    )

    history["config"] = {
        "num_byzantine": num_byz,
        "aggregation": agg,
        "epsilon": TARGET_EPSILON,
        "alpha": DIRICHLET_ALPHA,
        "description": desc,
    }

    all_full_results[exp_key] = history

    with open(full_results_path, 'w') as f:
        json.dump(all_full_results, f, indent=2)
    print(f"  Saved. Done: {len(all_full_results)}/{len(full_experiments)}")


# =============================================================================
# PHASE 12: CONSOLIDATE ALL RESULTS FROM ALL NOTEBOOKS
# =============================================================================

print("\n" + "=" * 60)
print("CONSOLIDATING ALL RESULTS FOR PAPER")
print("=" * 60)

# Load results from previous notebooks
prev_results = {}

# Notebook 1: Centralized baseline
nb1_path = os.path.join(OUTPUT_DIR, "centralized_results.json")
if os.path.exists(nb1_path):
    prev_results["nb1"] = load_json(nb1_path)
    print("  Loaded: Notebook 1 (Centralized)")

# Notebook 2: FedAvg
nb2_path = os.path.join(OUTPUT_DIR, "fedavg_final_results.json")
if os.path.exists(nb2_path):
    prev_results["nb2"] = load_json(nb2_path)
    print("  Loaded: Notebook 2 (FedAvg)")

# Notebook 3: DP-FedAvg
nb3_path = os.path.join(OUTPUT_DIR, "dp_fedavg_final_results.json")
if os.path.exists(nb3_path):
    prev_results["nb3"] = load_json(nb3_path)
    print("  Loaded: Notebook 3 (DP-FedAvg)")

# Notebook 4: SecAgg benchmarks
nb4_path = os.path.join(OUTPUT_DIR, "secagg_results.json")
if os.path.exists(nb4_path):
    prev_results["nb4"] = load_json(nb4_path)
    print("  Loaded: Notebook 4 (SecAgg)")

# Notebook 5: Byzantine
nb5_path = os.path.join(OUTPUT_DIR, "byzantine_final_results.json")
if os.path.exists(nb5_path):
    prev_results["nb5"] = load_json(nb5_path)
    print("  Loaded: Notebook 5 (Byzantine)")


# =============================================================================
# PHASE 13: PAPER RESULTS TABLE
# =============================================================================

print("\n" + "=" * 60)
print("TABLE FOR PAPER: COMPREHENSIVE RESULTS COMPARISON")
print("=" * 60)

print(f"\n  {'Configuration':<45} {'BLEU':>8} {'JSON%':>8} {'CmdAcc%':>10} {'ActAcc%':>10} {'Eps':>6}")
print(f"  {'='*45} {'='*8} {'='*8} {'='*10} {'='*10} {'='*6}")

# Row 1: Centralized baseline (NB1)
if "nb1" in prev_results:
    e = prev_results["nb1"]["evaluation"]
    print(f"  {'Centralized (upper bound)':<45} "
          f"{e['bleu_score_avg']:>8.4f} "
          f"{e['json_validity_rate']:>7.1f}% "
          f"{e['command_type_accuracy']:>9.1f}% "
          f"{e['action_accuracy']:>9.1f}% "
          f"{'--':>6}")

# Row 2: FedAvg only, alpha=0.5 (NB2)
if "nb2" in prev_results:
    exps = prev_results["nb2"].get("experiments", {})
    if "0.5" in exps:
        m = exps["0.5"]["final_metrics"]
        print(f"  {'FedAvg (a=0.5)':<45} "
              f"{m['bleu']:>8.4f} "
              f"{m['json_validity']:>7.1f}% "
              f"{m['cmd_type_acc']:>9.1f}% "
              f"{m['action_acc']:>9.1f}% "
              f"{'--':>6}")

# Row 3: FedAvg only, alpha=0.1 (NB2)
if "nb2" in prev_results:
    exps = prev_results["nb2"].get("experiments", {})
    if "0.1" in exps:
        m = exps["0.1"]["final_metrics"]
        print(f"  {'FedAvg (a=0.1, severe non-IID)':<45} "
              f"{m['bleu']:>8.4f} "
              f"{m['json_validity']:>7.1f}% "
              f"{m['cmd_type_acc']:>9.1f}% "
              f"{m['action_acc']:>9.1f}% "
              f"{'--':>6}")

# Rows 4-7: DP-FedAvg at different epsilons (NB3)
if "nb3" in prev_results:
    for eps_key in ["eps_1", "eps_4", "eps_8", "eps_16"]:
        exp_a = prev_results["nb3"].get("experiment_a", {})
        if eps_key in exp_a:
            m = exp_a[eps_key]["final_metrics"]
            eps_val = exp_a[eps_key]["epsilon"]
            print(f"  { 'FedAvg + DP (e='+str(eps_val)+', a=0.5)':<45} "
                  f"{m['bleu']:>8.4f} "
                  f"{m['json_validity']:>7.1f}% "
                  f"{m['cmd_type_acc']:>9.1f}% "
                  f"{m['action_acc']:>9.1f}% "
                  f"{eps_val:>6}")

# Rows: Full framework experiments (NB6)
print(f"  {'-'*45} {'-'*8} {'-'*8} {'-'*10} {'-'*10} {'-'*6}")
for num_byz, agg, desc in full_experiments:
    key = f"full_byz{num_byz}_{agg}_eps{TARGET_EPSILON}"
    if key in all_full_results and all_full_results[key].get("final_eval_500"):
        m = all_full_results[key]["final_eval_500"]
        print(f"  {desc:<45} "
              f"{m['bleu']:>8.4f} "
              f"{m['json_validity']:>7.1f}% "
              f"{m['cmd_type_acc']:>9.1f}% "
              f"{m['action_acc']:>9.1f}% "
              f"{TARGET_EPSILON:>6}")

# SecAgg overhead (NB4)
if "nb4" in prev_results:
    bench = prev_results["nb4"].get("benchmarks", {})
    if "5" in bench:
        overhead = bench["5"]["overhead_per_round_ms"]
        total20 = bench["5"]["total_overhead_20rounds_sec"]
        print(f"\n  SecAgg overhead (5 clients): {overhead:.1f} ms/round, "
              f"{total20:.1f}s over 20 rounds ({bench['5'].get('update_size_mb', 17.19)} MB/update)")


# =============================================================================
# PHASE 14: FINAL COMPARISON PLOTS
# =============================================================================

# Plot 1: Complete ablation bar chart
fig, ax = plt.subplots(figsize=(14, 6))

configs = []
bleu_vals = []
action_vals = []
bar_colors = []

# Centralized
if "nb1" in prev_results:
    configs.append("Centralized\n(upper bound)")
    bleu_vals.append(prev_results["nb1"]["evaluation"]["bleu_score_avg"])
    action_vals.append(prev_results["nb1"]["evaluation"]["action_accuracy"])
    bar_colors.append("#4CAF50")

# FedAvg only
if "nb2" in prev_results and "0.5" in prev_results["nb2"].get("experiments", {}):
    m = prev_results["nb2"]["experiments"]["0.5"]["final_metrics"]
    configs.append("FedAvg\n(a=0.5)")
    bleu_vals.append(m["bleu"])
    action_vals.append(m["action_acc"])
    bar_colors.append("#2196F3")

# FedAvg + DP (eps=8)
if "nb3" in prev_results:
    exp_a = prev_results["nb3"].get("experiment_a", {})
    if "eps_8" in exp_a:
        m = exp_a["eps_8"]["final_metrics"]
        configs.append("FedAvg+DP\n(e=8)")
        bleu_vals.append(m["bleu"])
        action_vals.append(m["action_acc"])
        bar_colors.append("#FF9800")

# Full framework entries
framework_colors = {
    "full_byz0_fedavg_eps8": "#9C27B0",
    "full_byz1_krum_eps8": "#E91E63",
    "full_byz1_trimmed_mean_eps8": "#00BCD4",
    "full_byz2_krum_eps8": "#F44336",
    "full_byz2_trimmed_mean_eps8": "#009688",
    "full_byz2_fedavg_eps8": "#795548",
}
framework_labels = {
    "full_byz0_fedavg_eps8": "DP+SecAgg\n(0 Byz)",
    "full_byz1_krum_eps8": "DP+SecAgg\n+Krum (1B)",
    "full_byz1_trimmed_mean_eps8": "DP+SecAgg\n+TrMean (1B)",
    "full_byz2_krum_eps8": "DP+SecAgg\n+Krum (2B)",
    "full_byz2_trimmed_mean_eps8": "DP+SecAgg\n+TrMean (2B)",
    "full_byz2_fedavg_eps8": "DP+FedAvg\n(2B, no def)",
}

for key in ["full_byz0_fedavg_eps8", "full_byz2_krum_eps8",
            "full_byz2_trimmed_mean_eps8", "full_byz2_fedavg_eps8"]:
    if key in all_full_results and all_full_results[key].get("final_eval_500"):
        m = all_full_results[key]["final_eval_500"]
        configs.append(framework_labels.get(key, key))
        bleu_vals.append(m["bleu"])
        action_vals.append(m["action_acc"])
        bar_colors.append(framework_colors.get(key, "#999999"))

if configs:
    x = np.arange(len(configs))
    width = 0.35

    bars1 = ax.bar(x - width/2, [b * 100 for b in bleu_vals], width,
                   label="BLEU x 100", color=bar_colors, alpha=0.85)
    bars2 = ax.bar(x + width/2, action_vals, width,
                   label="Action Acc %", color=bar_colors, alpha=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(configs, fontsize=9, ha='center')
    ax.set_ylabel("Score (%)", fontsize=12)
    ax.set_title("Complete Framework Ablation Study", fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 110)

plt.tight_layout()
plot1_path = os.path.join(OUTPUT_DIR, "full_framework_ablation.png")
plt.savefig(plot1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {plot1_path}")

# Plot 2: Convergence comparison (full framework)
fig, ax = plt.subplots(figsize=(10, 5))

plot_keys = [
    ("full_byz0_fedavg_eps8", "DP only (0 Byz)", "#4CAF50"),
    ("full_byz2_krum_eps8", "DP + Krum (2 Byz)", "#E91E63"),
    ("full_byz2_trimmed_mean_eps8", "DP + TrMean (2 Byz)", "#2196F3"),
    ("full_byz2_fedavg_eps8", "DP + FedAvg (2 Byz, no def)", "#FF5722"),
]

for key, label, color in plot_keys:
    if key in all_full_results:
        rounds = all_full_results[key]["rounds"]
        losses = all_full_results[key]["train_losses"]
        ax.plot(rounds, losses, 'o-', color=color, linewidth=2,
                markersize=4, label=label)

ax.set_xlabel("Communication Round", fontsize=12)
ax.set_ylabel("Average Training Loss", fontsize=12)
ax.set_title("Full Framework Convergence (DP + Byzantine)", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plot2_path = os.path.join(OUTPUT_DIR, "full_framework_convergence.png")
plt.savefig(plot2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {plot2_path}")


# =============================================================================
# PHASE 15: SAVE EVERYTHING
# =============================================================================

paper_results = {
    "timestamp": datetime.now().isoformat(),
    "full_framework_experiments": {},
    "previous_notebooks": {},
}

# Full framework
for key, val in all_full_results.items():
    paper_results["full_framework_experiments"][key] = {
        "config": val.get("config", {}),
        "final_eval_500": val.get("final_eval_500", {}),
        "dp_config": val.get("dp_config", {}),
        "time_minutes": val.get("total_time_minutes", 0),
        "convergence": {
            "rounds": val.get("rounds", []),
            "losses": val.get("train_losses", []),
        }
    }

# Previous notebook summaries
if "nb1" in prev_results:
    paper_results["previous_notebooks"]["centralized"] = {
        "bleu": prev_results["nb1"]["evaluation"]["bleu_score_avg"],
        "json_validity": prev_results["nb1"]["evaluation"]["json_validity_rate"],
        "cmd_type_acc": prev_results["nb1"]["evaluation"]["command_type_accuracy"],
        "action_acc": prev_results["nb1"]["evaluation"]["action_accuracy"],
    }

if "nb4" in prev_results:
    paper_results["previous_notebooks"]["secagg_overhead"] = {
        "5_clients_ms_per_round": prev_results["nb4"]["benchmarks"]["5"]["overhead_per_round_ms"],
        "5_clients_total_20rounds_sec": prev_results["nb4"]["benchmarks"]["5"]["total_overhead_20rounds_sec"],
        "update_size_mb": prev_results["nb4"]["benchmarks"]["5"]["update_size_mb"],
    }

paper_path = os.path.join(OUTPUT_DIR, "paper_results_consolidated.json")
with open(paper_path, 'w') as f:
    json.dump(paper_results, f, indent=2)
print(f"\nConsolidated paper results: {paper_path}")


# =============================================================================
# PHASE 16: SUMMARY
# =============================================================================

print(f"\n{'='*60}")
print(f"NOTEBOOK 6 COMPLETE — FULL FRAMEWORK EVALUATION")
print(f"{'='*60}")
print(f"End time: {datetime.now().strftime('%H:%M:%S')}")
print(f"")
print(f"Files in {OUTPUT_DIR}/:")
print(f"  full_framework_results.json         (raw experiment data)")
print(f"  paper_results_consolidated.json     (all results for paper)")
print(f"  full_framework_ablation.png         (ablation bar chart)")
print(f"  full_framework_convergence.png      (convergence curves)")
print(f"")
print(f"FULL FRAMEWORK RESULTS (500-sample eval):")
for num_byz, agg, desc in full_experiments:
    key = f"full_byz{num_byz}_{agg}_eps{TARGET_EPSILON}"
    if key in all_full_results and all_full_results[key].get("final_eval_500"):
        m = all_full_results[key]["final_eval_500"]
        print(f"  {desc}: BLEU={m['bleu']:.4f} | "
              f"JSON={m['json_validity']}% | "
              f"CmdType={m['cmd_type_acc']}% | "
              f"Action={m['action_acc']}%")

print(f"\n{'='*60}")
print(f"ALL EXPERIMENTS COMPLETE. Ready to write the paper.")
print(f"{'='*60}")

Mounted at /content/drive
Installing packages...
All packages installed.
Extracting dataset...
Dataset: /content/dataset/final_dataset
All imports successful.

GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB
Train: 12033 | Val: 1203 | Test: 1210

Loading model (bf16 + SDPA for H100)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Trainable (LoRA): 4,505,600

Splitting data...
  Client 0: 1753 samples
  Client 1: 2122 samples
  Client 2: 3248 samples
  Client 3: 2374 samples
  Client 4: 2536 samples

RUNNING FULL FRAMEWORK EXPERIMENTS (DP + Byzantine)
Start time: 04:06:23
  Loaded: full_byz0_fedavg_eps8
  Loaded: full_byz1_krum_eps8
  Loaded: full_byz1_trimmed_mean_eps8
  Loaded: full_byz2_krum_eps8
  Loaded: full_byz2_trimmed_mean_eps8
  Loaded: full_byz2_fedavg_eps8

  full_byz0_fedavg_eps8 already done. Skipping.

  full_byz1_krum_eps8 already done. Skipping.

  full_byz1_trimmed_mean_eps8 already done. Skipping.

  full_byz2_krum_eps8 already done. Skipping.

  full_byz2_trimmed_mean_eps8 already done. Skipping.

  full_byz2_fedavg_eps8 already done. Skipping.

CONSOLIDATING ALL RESULTS FOR PAPER
  Loaded: Notebook 1 (Centralized)
  Loaded: Notebook 2 (FedAvg)
  Loaded: Notebook 3 (DP-FedAvg)
  Loaded: Notebook 4 (SecAgg)
  Loaded: Notebook 5 (Byzantine)

TABLE FOR PAPER: COMPREHENSIVE RESULTS COMPARISON

  